# Acurácia na avaliação por similaridade

Este notebook cria uma figura para cada modelo de linguagem. Cada figura contém um subplot por dataset e uma curva para cada condição experimental disponível.

A similaridade é dividida em quantis para que cada ponto represente várias questões. O eixo x mostra a similaridade média da faixa e o eixo y mostra a métrica escolhida em PLOT_METRIC. A tabela de cobertura registra os totais originais e os selecionados. Os filtros nunca alteram o arquivo original.

Condições dos estudantes: baseline, autorreflexão simples/complexa e reflexão externa simples/complexa. Para o GPT-5.4 Petrobras aparecem apenas baseline e as duas condições de autorreflexão, pois ele não possui condições de reflexão externa.

Os eixos Y se ajustam aos dados, com margem configurável no início. Compare os valores e as marcações do eixo: as barras podem começar acima de zero. No teste, as vistas completa e filtrada compartilham os limites por modelo/dataset e tipo de gráfico.

In [ ]:
from pathlib import Path
import math
import json
import sys
import re

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import PercentFormatter

plt.style.use("seaborn-v0_8-whitegrid")

EXPERIMENT_ID = "91ccab5e5028"
N_SIMILARITY_BINS = 10
SAVE_PDF = True
# Escolha filtros antes de executar as células seguintes.
EXCLUDE_FLAGS = []  # ex.: ["length_exhausted", "partial_think", "context_exceeded"]
EXCLUDE_METHODS = []  # ex.: ["judge"]
CONDITIONS = []  # vazio = todas as condições existentes
RACE_SUBSETS = []  # vazio = middle + high; ou ["middle"] / ["high"]
PAIRED = False  # interseção das condições por modelo e questão
RESOLVED_ONLY = False  # com PAIRED=True, compara apenas questões resolvidas em todas
PLOT_METRIC = "accuracy_all"  # ou "accuracy": somente respostas resolvidas
FILTER_CONFIG = dict(exclude_flags=EXCLUDE_FLAGS, exclude_methods=EXCLUDE_METHODS,
                     conditions=CONDITIONS, race_subsets=RACE_SUBSETS,
                     paired=PAIRED, resolved_only=RESOLVED_ONLY)

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "run_experiment.py").exists() and (candidate / "rmcq").is_dir():
            return candidate
    raise FileNotFoundError("Não encontrei a raiz do repositório Reflection-MCQ.")

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from rmcq.analysis import filter_outcomes
from run_experiment import load_jsonl, json_hash, save_json, save_csv
RESULT_DIR = ROOT / "data" / "results" / "reflection_top1" / EXPERIMENT_ID
OUTCOMES_PATH = RESULT_DIR / "analysis" / "all_outcomes.jsonl"
PLOTS_DIR = RESULT_DIR / "analysis" / "views" / json_hash({**FILTER_CONFIG, "metric": PLOT_METRIC}) / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repositório: {ROOT}")
print(f"Resultados:  {OUTCOMES_PATH}")
print(f"Figuras:     {PLOTS_DIR}")

# Eixo Y: ajuste automático por dataset; margem em pontos de acurácia.
AUTO_YLIM = True  # False restaura 0–100%
YLIM_PADDING = 0.04
YLIM_MIN_SPAN = 0.12


In [ ]:
if not OUTCOMES_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {OUTCOMES_PATH}\n"
        "Execute o estágio finish no servidor GPU ou copie a pasta final de resultados."
    )

raw_rows = load_jsonl(OUTCOMES_PATH)
filtered_rows, filter_audit = filter_outcomes(raw_rows, **FILTER_CONFIG)
save_json(PLOTS_DIR.parent / "filters.json", {**FILTER_CONFIG, "metric": PLOT_METRIC})
save_csv(PLOTS_DIR.parent / "coverage.csv", filter_audit)
raw_results = pd.DataFrame(raw_rows)
if not filtered_rows:
    raise ValueError("Nenhum item passou pelos filtros; consulte coverage.csv.")
results = pd.DataFrame(filtered_rows)
EVAL_SPLITS = sorted({row.get("eval_split", "validation") for row in raw_rows})
if len(EVAL_SPLITS) != 1:
    raise ValueError("Analise um split por vez.")
EVAL_LABEL = "teste" if EVAL_SPLITS == ["test"] else "validação"
if PLOT_METRIC not in {"accuracy", "accuracy_all"}:
    raise ValueError("PLOT_METRIC deve ser accuracy ou accuracy_all")
required_columns = {"model", "dataset", "condition", "val_uid", "similarity", "correct"}
missing_columns = sorted(required_columns - set(results.columns))
if missing_columns:
    raise ValueError(f"Colunas ausentes em all_outcomes.jsonl: {missing_columns}")

results["similarity"] = pd.to_numeric(results["similarity"], errors="coerce")
# Em resultados novos, `correct` já pode vir como 1.0/0.0/NaN.
results["correct_value"] = pd.to_numeric(results["correct"], errors="coerce")

print(f"Linhas: {len(results):,}")
print(f"Modelos: {', '.join(map(str, results['model'].dropna().unique()))}")
print(f"Datasets: {', '.join(map(str, results['dataset'].dropna().unique()))}")
print(f"Resolvidas (correct_value não nulo): {results['correct_value'].notna().sum():,}")
results.head(3)

In [ ]:
CONDITION_ORDER = [
    "baseline",
    "self_simple",
    "self_complex",
    "teacher_simple",
    "teacher_complex",
]

CONDITION_LABELS = {
    "baseline": "Baseline",
    "self_simple": "Self-ref. simples",
    "self_complex": "Self-ref. complexa",
    "teacher_simple": "Ref. externa simples",
    "teacher_complex": "Ref. externa complexa",
}

CONDITION_COLORS = {
    "baseline": "#222222",
    "self_simple": "#1f77b4",
    "self_complex": "#17becf",
    "teacher_simple": "#d62728",
    "teacher_complex": "#ff7f0e",
}

DATASET_ORDER = ["aqua", "arc", "logiqa2", "openbookqa", "race"]
DATASET_LABELS = {
    "aqua": "AQuA",
    "arc": "ARC",
    "logiqa2": "LogiQA 2.0",
    "openbookqa": "OpenBookQA",
    "race": "RACE",
}

MODEL_LABELS = {
    "phi2": "Phi-2",
    "phi4-mini": "Phi-4-mini-instruct",
    "mistral-7b-instruct": "Mistral-7B-Instruct-v0.3",
    "qwen3-8b": "Qwen3-8B (sem thinking)",
    "deepseek-r1-distill-llama-8b": "DeepSeek-R1-Distill-Llama-8B",
    "llama3.1-8b": "Llama 3.1 8B",
    "gpt-5-4-petrobras": "GPT-5.4 Petrobras",
}

# Uma única atribuição de faixas por dataset mantém condições e modelos alinhados.
pair_similarity = (
    raw_results.loc[raw_results["similarity"].notna(), ["dataset", "val_uid", "similarity"]]
    .drop_duplicates(["dataset", "val_uid"])
    .copy()
)

def assign_quantile_bins(group: pd.DataFrame) -> pd.DataFrame:
    group = group.copy()
    n_bins = min(N_SIMILARITY_BINS, group["similarity"].nunique(), len(group))
    if n_bins < 2:
        group["similarity_bin"] = 0
    else:
        group["similarity_bin"] = pd.qcut(
            group["similarity"], q=n_bins, labels=False, duplicates="drop"
        ).astype(int)
    return group

pair_similarity = pd.concat(
    [
        assign_quantile_bins(group.drop(columns="dataset")).assign(dataset=dataset)
        for dataset, group in pair_similarity.groupby("dataset", sort=False)
    ],
    ignore_index=True,
)

plot_rows = results.merge(
    pair_similarity[["dataset", "val_uid", "similarity_bin"]],
    on=["dataset", "val_uid"],
    how="inner",
    validate="many_to_one",
)

bin_centers = (
    pair_similarity.groupby(["dataset", "similarity_bin"], as_index=False)
    .agg(similarity=("similarity", "mean"), n_questions=("val_uid", "nunique"))
)

binned_accuracy = (
    plot_rows.groupby(["model", "dataset", "condition", "similarity_bin"], as_index=False)
    .agg(accuracy=("correct_value", "mean"), n_resolved=("correct_value", "count"),
         n_selected=("val_uid", "size"), n_correct=("correct_value", "sum"))
    .merge(bin_centers, on=["dataset", "similarity_bin"], how="left")
)

binned_accuracy["accuracy_all"] = binned_accuracy["n_correct"] / binned_accuracy["n_selected"]
binned_accuracy.to_csv(PLOTS_DIR.parent / "binned_accuracy.csv", index=False)
binned_accuracy.head()

In [ ]:
coverage = pd.DataFrame(filter_audit)
coverage.style.format({"accuracy": "{:.1%}", "accuracy_all": "{:.1%}",
                       "coverage_original": "{:.1%}", "coverage_selected": "{:.1%}"})


In [ ]:
from rmcq.plotting import accuracy_ylim

def conditions_for_model(model: str) -> list[str]:
    available = set(plot_rows.loc[plot_rows["model"] == model, "condition"])
    if model == "gpt-5-4-petrobras" or model.casefold().startswith("gpt-5"):
        desired = ["baseline", "self_simple", "self_complex"]
    else:
        desired = CONDITION_ORDER
    return [condition for condition in desired if condition in available]

def slug(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", value.casefold()).strip("-")

def plot_model(model: str, save: bool = True):
    model_data = binned_accuracy[binned_accuracy["model"] == model]
    available_datasets = set(model_data["dataset"])
    datasets = [d for d in DATASET_ORDER if d in available_datasets]
    datasets += sorted(available_datasets - set(datasets))
    conditions = conditions_for_model(model)

    if not datasets:
        raise ValueError(f"Não há resultados com similaridade para o modelo {model!r}.")

    ncols = 2
    nrows = math.ceil(len(datasets) / ncols)
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(13, 4.6 * nrows), sharey=False, squeeze=False
    )
    legend_handles = {}

    for ax, dataset in zip(axes.flat, datasets):
        subset = model_data[model_data["dataset"] == dataset]
        for condition in conditions:
            line_data = subset[subset["condition"] == condition].sort_values("similarity")
            line_data = line_data[line_data[PLOT_METRIC].notna()]
            if line_data.empty:
                continue
            line, = ax.plot(
                line_data["similarity"],
                line_data[PLOT_METRIC],
                color=CONDITION_COLORS[condition],
                marker="o",
                markersize=4.5,
                linewidth=2,
                label=CONDITION_LABELS[condition],
            )
            legend_handles[condition] = line

        ax.set_title(DATASET_LABELS.get(dataset, dataset))
        ax.set_xlabel("Similaridade média no quantil")
        ax.set_ylabel("Acertos / selecionadas" if PLOT_METRIC == "accuracy_all" else "Acurácia entre resolvidas")
        ax.set_ylim(*(accuracy_ylim(subset[PLOT_METRIC], padding=YLIM_PADDING, min_span=YLIM_MIN_SPAN) if AUTO_YLIM else (0, 1)))
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))
        ax.grid(alpha=0.25)

    for ax in axes.flat[len(datasets):]:
        ax.set_visible(False)

    ordered_handles = [legend_handles[c] for c in conditions if c in legend_handles]
    ordered_labels = [CONDITION_LABELS[c] for c in conditions if c in legend_handles]
    if ordered_handles:
        fig.legend(
            ordered_handles, ordered_labels, loc="lower center", ncol=len(ordered_handles),
            bbox_to_anchor=(0.5, 0.01), frameon=False,
        )

    title = MODEL_LABELS.get(model, model)
    metric_label = "Acertos / selecionadas" if PLOT_METRIC == "accuracy_all" else "Acertos / resolvidas"
    fig.suptitle(f"{title}: {metric_label} no {EVAL_LABEL} por similaridade", fontsize=15, y=0.995)
    fig.tight_layout(rect=(0, 0.075, 1, 0.97))

    if save:
        png_path = PLOTS_DIR / f"accuracy_by_similarity_{slug(model)}.png"
        fig.savefig(png_path, dpi=180, bbox_inches="tight")
        if SAVE_PDF:
            fig.savefig(png_path.with_suffix(".pdf"), bbox_inches="tight")
        print(f"Salvo: {png_path}")

    plt.show()
    return fig


In [ ]:
preferred_models = [
    "phi2",
    "deepseek-r1-distill-llama-8b",
    "llama3.1-8b",
    "gpt-5-4-petrobras",
]
available_models = list(map(str, results["model"].dropna().unique()))
model_order = [model for model in preferred_models if model in available_models]
model_order += sorted(set(available_models) - set(model_order))

figures = {}
for model in model_order:
    figures[model] = plot_model(model)

## Leitura dos gráficos

- `accuracy_all`: acertos divididos por todos os itens selecionados, incluindo não resolvidos no denominador.
- `accuracy`: acertos divididos somente pelas respostas resolvidas.
- `coverage.csv` mantém `n_original`, `n_selected`, `n_excluded` e as duas coberturas.
- `PAIRED=True` exige presença nas mesmas condições por modelo/questão; combine com `RESOLVED_ONLY=True` para a interseção de respostas resolvidas.
- Os quantis são definidos no conjunto original e permanecem fixos ao mudar filtros.
- Filtrar não recupera uma reflexão descartada na geração. O teste contrafactual de usá-la exigiria outra execução.
- Cada seleção salva figuras, métricas e configuração em uma pasta própria, sem modificar a run.
